In [1]:
!pip install -q transformers datasets accelerate scikit-learn pandas numpy joblib

In [2]:
import os
import numpy as np
import pandas as pd
import torch
import joblib

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, jaccard_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

In [3]:
import pandas as pd

data_path = "/kaggle/input/datasets/overgame1234/merged-text-tags-grouped/merged_text_tags_grouped.csv"
df = pd.read_csv(data_path)

In [4]:
df = df[["text", "Tags"]].copy()
df = df.dropna(subset=["text", "Tags"])

df["text"] = df["text"].astype(str).str.strip()
df["Tags"] = df["Tags"].astype(str).str.strip()

df = df[(df["text"] != "") & (df["Tags"] != "")]



In [5]:
df["label_list"] = df["Tags"].apply(lambda x: x.split())
df[["text", "Tags", "label_list"]].head()  


,text,Tags,label_list
0,# + items .append is not a function codeblock ...,javascript,[javascript]
1,# - how to parallel code that lock several obj...,c#,[c#]
2,# . what do and # do in this code codeblock tr...,javascript,[javascript]
3,# .dialog is not a function error after using ...,javascript,[javascript]
4,# .dialog is not a function error i am trying ...,javascript,[javascript]


In [6]:
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df["label_list"])

print("Number of labels:", len(mlb.classes_))
print("y shape:", y.shape)
print("First labels:", mlb.classes_[:20])

Number of labels: 50
y shape: (1879950, 50)
First labels: ['active-directory' 'algorithm' 'amazon-ec2' 'android' 'apache' 'api'
 'architecture' 'bash' 'c#' 'c++' 'centos' 'data-structures'
 'database-design' 'debugging' 'design-patterns' 'dns' 'ftp' 'git' 'http'
 'image-processing']


In [7]:
X = df["text"].tolist()

In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42
)

print("Train:", len(X_train), y_train.shape)
print("Val  :", len(X_val), y_val.shape)
print("Test :", len(X_test), y_test.shape)

Train: 1503960 (1503960, 50)
Val  : 187995 (187995, 50)
Test : 187995 (187995, 50)


In [9]:
from datasets import Dataset
import numpy as np
y_train = y_train.astype(np.float32)
y_val = y_val.astype(np.float32)
y_test = y_test.astype(np.float32)
train_size = min(400000, len(X_train))
val_size = min(50000, len(X_val))
test_size = min(50000, len(X_test))

rng = np.random.RandomState(42)

train_idx = rng.choice(len(X_train), train_size, replace=False)
val_idx = rng.choice(len(X_val), val_size, replace=False)
test_idx = rng.choice(len(X_test), test_size, replace=False)

X_train_sub = X_train.iloc[train_idx].tolist() if hasattr(X_train, "iloc") else [X_train[i] for i in train_idx]
X_val_sub = X_val.iloc[val_idx].tolist() if hasattr(X_val, "iloc") else [X_val[i] for i in val_idx]
X_test_sub = X_test.iloc[test_idx].tolist() if hasattr(X_test, "iloc") else [X_test[i] for i in test_idx]

y_train_sub = y_train[train_idx].tolist()
y_val_sub = y_val[val_idx].tolist()
y_test_sub = y_test[test_idx].tolist()

train_dataset = Dataset.from_dict({
    "text": X_train_sub,
    "labels": y_train_sub
})

val_dataset = Dataset.from_dict({
    "text": X_val_sub,
    "labels": y_val_sub
})

test_dataset = Dataset.from_dict({
    "text": X_test_sub,
    "labels": y_test_sub
})

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 400000
Val: 50000
Test: 50000


In [10]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/400000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [12]:
num_labels = y_train.shape[1]

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

print("Num labels:", num_labels)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Num labels: 50


In [13]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/distilbert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_at_3",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

In [14]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

def top_k_binary_predictions(y_scores, k):
    y_pred = np.zeros_like(y_scores, dtype=int)
    topk_idx = np.argsort(-y_scores, axis=1)[:, :k]

    for i in range(y_scores.shape[0]):
        y_pred[i, topk_idx[i]] = 1

    return y_pred

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    results = {}

    for k in [1, 2, 3, 4]:
        preds_k = top_k_binary_predictions(logits, k)

        results[f"precision_at_{k}"] = precision_score(labels, preds_k, average="micro", zero_division=0)
        results[f"recall_at_{k}"] = recall_score(labels, preds_k, average="micro", zero_division=0)
        results[f"f1_at_{k}"] = f1_score(labels, preds_k, average="micro", zero_division=0)

    return results

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [17]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision At 1,Recall At 1,F1 At 1,Precision At 2,Recall At 2,F1 At 2,Precision At 3,Recall At 3,F1 At 3,Precision At 4,Recall At 4,F1 At 4
1,0.069845,0.047310,0.854060,0.753046,0.800379,0.503500,0.887897,0.642601,0.350747,0.927787,0.509049,0.268785,0.947978,0.418820
2,0.044716,0.044435,0.863800,0.761634,0.809506,0.508780,0.897208,0.649339,0.353807,0.935881,0.513490,0.270745,0.954891,0.421874
3,0.039180,0.043971,0.867160,0.764597,0.812655,0.510240,0.899783,0.651203,0.354607,0.937997,0.514651,0.271430,0.957307,0.422941
4,0.035302,0.044192,0.867900,0.765249,0.813349,0.510040,0.899430,0.650947,0.354647,0.938103,0.514709,0.271350,0.957025,0.422817


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=50000, training_loss=0.047260720825195314, metrics={'train_runtime': 7650.2096, 'train_samples_per_second': 209.145, 'train_steps_per_second': 6.536, 'total_flos': 2.6516158464e+16, 'train_loss': 0.047260720825195314, 'epoch': 4.0})

In [20]:
val_results = trainer.evaluate(eval_dataset=val_dataset)
print("Validation Results:", val_results)

test_results = trainer.evaluate(eval_dataset=test_dataset)
print("Test Results:", test_results)

Validation Results: {'eval_loss': 0.044192250818014145, 'eval_precision_at_1': 0.8679, 'eval_recall_at_1': 0.7652494401043963, 'eval_f1_at_1': 0.8133487025218589, 'eval_precision_at_2': 0.51004, 'eval_recall_at_2': 0.8994304054173207, 'eval_f1_at_2': 0.6509473093097309, 'eval_precision_at_3': 0.35464666666666667, 'eval_recall_at_3': 0.9381028797150264, 'eval_f1_at_3': 0.5147092261026477, 'eval_precision_at_4': 0.27135, 'eval_recall_at_4': 0.9570247059445923, 'eval_f1_at_4': 0.42281667426287556, 'eval_runtime': 85.3227, 'eval_samples_per_second': 586.01, 'eval_steps_per_second': 18.319, 'epoch': 4.0}


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Test Results: {'eval_loss': 0.04461038485169411, 'eval_precision_at_1': 0.8663, 'eval_recall_at_1': 0.7627491723603578, 'eval_f1_at_1': 0.8112334719256845, 'eval_precision_at_2': 0.50992, 'eval_recall_at_2': 0.8979361837007819, 'eval_f1_at_2': 0.6504579432099395, 'eval_precision_at_3': 0.35527333333333333, 'eval_recall_at_3': 0.9384200887511446, 'eval_f1_at_3': 0.5154167553242934, 'eval_precision_at_4': 0.27178, 'eval_recall_at_4': 0.9571740508558146, 'eval_f1_at_4': 0.42335311618923005, 'eval_runtime': 84.2889, 'eval_samples_per_second': 593.198, 'eval_steps_per_second': 18.543, 'epoch': 4.0}
